Import libraries and load data 

In [8]:
import pandas as pd

In [9]:
df = pd.read_csv('Merged_Energy_Weather_30min.csv')
df

,Datetime,Consumption_MW,Forecast_DayMinus1_MW,Forecast_Day_MW,Hour,Minute,DayOfWeek,Month,DayOfYear,IsWeekend,...,Lag_672,Temp_CC,Temp_CE,Temp_CW,Temp_NC,Temp_NE,Temp_NW,Temp_SC,Temp_SE,Temp_SW
0,2018-01-15 00:00:00,66855.0,65200.0,65800.0,0,0,0,1,15,0,...,61127.0,4.266667,1.233333,6.925,2.385714,0.25,6.52,5.771429,6.366667,6.2
1,2018-01-15 00:30:00,65511.0,63600.0,64300.0,0,30,0,1,15,0,...,59962.0,4.266667,1.233333,6.925,2.385714,0.25,6.52,5.771429,6.366667,6.2
2,2018-01-15 01:00:00,63056.0,61500.0,62000.0,1,0,0,1,15,0,...,57879.0,4.266667,1.233333,6.925,2.385714,0.25,6.52,5.771429,6.366667,6.2
3,2018-01-15 01:30:00,62825.0,62400.0,63000.0,1,30,0,1,15,0,...,57901.0,4.266667,1.233333,6.925,2.385714,0.25,6.52,5.771429,6.366667,6.2
4,2018-01-15 02:00:00,62196.0,62300.0,62300.0,2,0,0,1,15,0,...,57261.0,4.266667,1.233333,6.925,2.385714,0.25,6.52,5.771429,6.366667,6.2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
84278,2022-12-31 19:00:00,50683.0,50100.0,49700.0,19,0,5,12,365,1,...,76622.0,13.116667,13.800000,15.000,14.600000,14.35,14.72,11.800000,13.480000,13.8
84279,2022-12-31 19:30:00,50304.0,49200.0,48800.0,19,30,5,12,365,1,...,76365.0,13.116667,13.800000,15.000,14.600000,14.35,14.72,11.800000,13.480000,13.8
84280,2022-12-31 20:00:00,49112.0,47700.0,47300.0,20,0,5,12,365,1,...,75452.0,13.116667,13.800000,15.000,14.600000,14.35,14.72,11.800000,13.480000,13.8
84281,2022-12-31 20:30:00,47715.0,46200.0,45900.0,20,30,5,12,365,1,...,74198.0,13.116667,13.800000,15.000,14.600000,14.35,14.72,11.800000,13.480000,13.8


In [10]:
df.columns

Index(['Datetime', 'Consumption_MW', 'Forecast_DayMinus1_MW',
       'Forecast_Day_MW', 'Hour', 'Minute', 'DayOfWeek', 'Month', 'DayOfYear',
       'IsWeekend', 'Lag_1', 'Lag_4', 'Lag_96', 'Lag_672', 'Temp_CC',
       'Temp_CE', 'Temp_CW', 'Temp_NC', 'Temp_NE', 'Temp_NW', 'Temp_SC',
       'Temp_SE', 'Temp_SW'],
      dtype='object')

LSTM

In [11]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout

# 1. Prepare Features
features = ['Consumption_MW', 'Hour', 'DayOfWeek', 'IsWeekend', 
            'Lag_1', 'Lag_4', 'Temp_CC', 'Temp_NE', 'Temp_NW'] # Add all your Temp cols
data = df[features].values

# 2. Scale the data
scaler = MinMaxScaler(feature_range=(0, 1))
scaled_data = scaler.fit_transform(data)

# 3. Create Sequences (Sliding Window)
def create_sequences(data, window_size):
    X, y = [], []
    for i in range(window_size, len(data)):
        X.append(data[i-window_size:i, :]) # Past 'window_size' steps
        y.append(data[i, 0]) # Target is Consumption_MW (index 0)
    return np.array(X), np.array(y)

WINDOW_SIZE = 48 # Looking back 24 hours (48 * 30 mins)
X, y = create_sequences(scaled_data, WINDOW_SIZE)

# Split into Train/Test (Don't shuffle! Time order matters)
train_size = int(len(X) * 0.8)
X_train, X_test = X[:train_size], X[train_size:]
y_train, y_test = y[:train_size], y[train_size:]

# 4. Build the LSTM Model
model = Sequential([
    # Input shape: (window_size, number_of_features)
    LSTM(units=64, return_sequences=True, input_shape=(X_train.shape, X_train.shape)),
    Dropout(0.2),
    LSTM(units=32, return_sequences=False),
    Dropout(0.2),
    Dense(units=16, activation='relu'),
    Dense(units=1) # Final output: Predicted Consumption
])

model.compile(optimizer='adam', loss='mean_squared_error')

# 5. Train
history = model.fit(X_train, y_train, epochs=20, batch_size=32, validation_split=0.1)

/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


ValueError: Invalid dtype: tuple